# Actividad 4: Aplicación de algoritmos de aprendizaje no supervisado con PySpark

**Materia:** Análisis de grandes volúmenes de datos  
**Institución:** Tecnológico de Monterrey, Posgrados  
**Autor:** Jonathan Javier Monsalve Giraldo (A01840272)  
**Profesores:** Dr. Iván Olmos Pineda, Luis Daniel Mendoza  
**Fecha:** 1 de junio de 2026  
**Dataset:** NYC TLC Yellow Taxi Trip Records 2024-2025  
**Modalidad:** Individual


## Objetivo

Aplicar algoritmos de aprendizaje no supervisado en PySpark MLlib sobre una muestra M' derivada de la muestra estratificada M construida en la Etapa 2 del proyecto del equipo. El problema elegido es la segmentación de viajes Yellow Taxi NYC en arquetipos operativos, sin utilizar una variable objetivo, para descubrir perfiles naturales de viaje según distancia, duración, velocidad, zona de origen, horario, pasajeros y régimen Flex Fare.

## Estructura del notebook

1. **Introducción**: aprendizaje no supervisado, algoritmos representativos y los disponibles en PySpark MLlib.
2. **Selección de los datos**: reconstrucción compacta de M, construcción de la muestra individual M' y validación de representatividad.
3. **Preparación del conjunto de entrenamiento y prueba**: partición estratificada train/test y validación del split como revisión de estabilidad.
4. **Construcción de modelos de aprendizaje no supervisado**: selección de features, pipeline, barrido de k, KMeans, comparación con GMM, evaluación e interpretación de arquetipos.

### Nota para el profesor

Las secciones 2.0, 2.1 y 2.2 reutilizan de forma compacta la reconstrucción de M desde la Etapa 2, la construcción de M' y la validación de representatividad usadas en la Actividad 3. Se conservan para que el notebook sea autocontenido y reproducible. El aporte nuevo de esta actividad inicia en la **Sección 3**, donde el split se usa para medir estabilidad del patrón, y sobre todo en la **Sección 4**, donde se derivan las variables operativas para clustering y se entrenan e interpretan los modelos no supervisados.


## 1. Introducción

### 1.1 Aprendizaje no supervisado

El aprendizaje no supervisado agrupa métodos que buscan estructura interna en datos sin una variable objetivo conocida. A diferencia del aprendizaje supervisado, donde cada observación incluye una etiqueta para entrenar y evaluar predicciones, aquí el modelo identifica similitudes, patrones, componentes latentes o casos poco usuales a partir de las variables disponibles.

La calidad de un resultado no supervisado no se mide comparando contra una respuesta verdadera, sino con criterios internos y con interpretación del dominio. Por ello, métricas como cohesión, separación, varianza explicada, probabilidad de pertenencia o frecuencia de patrones se usan como guías, pero deben complementarse con una lectura razonada de los grupos o estructuras descubiertas.

### 1.2 Familias representativas

Las técnicas no supervisadas se organizan en varias familias. El **clustering** agrupa observaciones similares; KMeans representa cada grupo por un centroide, BisectingKMeans construye divisiones jerárquicas y Gaussian Mixture Model permite pertenencia probabilística a componentes gaussianos. La **reducción de dimensión**, como PCA o SVD, resume muchas variables en menos componentes que conservan variabilidad. Las **reglas de asociación**, como FPGrowth, descubren combinaciones frecuentes de ítems o eventos. En **texto**, LDA identifica temas latentes a partir de vectores de conteos. La **detección de anomalías** puede aproximarse con distancia al centroide, baja probabilidad de pertenencia o reglas de negocio cuando no existe una etiqueta de fraude o error.

### 1.3 Algoritmos disponibles en PySpark MLlib

PySpark expone estas técnicas mediante la API moderna `pyspark.ml`, basada en DataFrames y en el patrón `Estimator`/`Transformer`/`Pipeline`. Un `Estimator` aprende parámetros con `fit()`, un `Transformer` agrega columnas con `transform()`, y un `Pipeline` encadena preparación de datos y modelo para aplicar en test las transformaciones aprendidas en train.

| Familia | Submódulo PySpark | Implementaciones relevantes | Uso típico |
|---|---|---|---|
| Clustering | `pyspark.ml.clustering` | `KMeans`, `BisectingKMeans`, `GaussianMixture`, `LDA`, `PowerIterationClustering` | Segmentación tabular, temas en texto o clusters sobre grafos |
| Reducción de dimensión | `pyspark.ml.feature` | `PCA` | Compresión de variables, visualización o preprocesamiento antes de clustering |
| Patrones frecuentes | `pyspark.ml.fpm` | `FPGrowth`, `PrefixSpan` | Canastas, secuencias y combinaciones frecuentes de eventos |
| Evaluación | `pyspark.ml.evaluation` | `ClusteringEvaluator` | Cálculo de silhouette como métrica interna de cohesión y separación |

La tabla resume las clases de alto nivel disponibles en la API DataFrame `pyspark.ml`, no las clases auxiliares de modelo, resumen o la API RDD antigua `pyspark.mllib`. La documentación oficial de Spark lista en clustering `KMeans`, `LDA`, `BisectingKMeans`, `GaussianMixture` y `PowerIterationClustering`; en patrones frecuentes, `FPGrowth` y `PrefixSpan`; y en transformación de features, `PCA` para reducción de dimensión. KMeans, BisectingKMeans y GMM producen una columna de cluster y pueden evaluarse con `ClusteringEvaluator` mediante silhouette. En no supervisado esta métrica no equivale a una verdad externa: se usa como criterio interno junto con tamaño de clusters, estabilidad train/test e interpretación de los perfiles.

### 1.4 Referencias

Apache Software Foundation. (2026). *Clustering - Spark 4.1.2 Documentation*. https://spark.apache.org/docs/latest/ml-clustering.html

Apache Software Foundation. (2026). *Extracting, transforming and selecting features - Spark 4.1.2 Documentation*. https://spark.apache.org/docs/latest/ml-features.html

Apache Software Foundation. (2026). *Frequent Pattern Mining - Spark 4.1.2 Documentation*. https://spark.apache.org/docs/latest/ml-frequent-pattern-mining.html

Polak, A. (2023). *Scaling machine learning with Spark: Distributed ML with MLlib, TensorFlow, and PyTorch*. O'Reilly Media. Capítulo 6, "Training Models with Spark MLlib".


## 2. Selección de los datos

### 2.0 Reconstrucción compacta de la muestra M (recap de Etapa 2)

> **Nota al profesor:** las secciones 2.0, 2.1 y 2.2 reutilizan de forma compacta la reconstrucción de la muestra M desde la Etapa 2, la construcción de M' y la validación de representatividad utilizadas en la Actividad 3. Se conservan para que este notebook sea autocontenido y reproducible. Si ya revisó esa parte en la entrega anterior, puede saltar la lectura detallada de estas secciones y continuar en la Sección 3, donde se reencuadra el split para aprendizaje no supervisado, y en la Sección 4, donde inicia el modelado de clustering.

La reconstrucción compacta mantiene el mismo criterio de la Actividad 3: carga de los 24 parquets mensuales, downcast del esquema, filtros destructivos de Etapa 2, imputaciones documentadas, construcción del estrato compuesto y extracción de M con `sampleBy` y piso mínimo por estrato.


In [ ]:
# Dependencias de Python para el notebook. Idempotente.
!pip install -q pyspark findspark pandas matplotlib

# Solo en Google Colab: descomentar para instalar Java (JVM de Spark).
# Localmente esta línea no es necesaria si Java ya está instalado.
# !apt-get install openjdk-8-jdk-headless -qq > /dev/null


In [ ]:
# Setup, descarga idempotente, lectura, downcast, filtros e imputaciones.
import findspark
findspark.init()

from pathlib import Path
import urllib.request

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.debug.maxToStringFields", 100)
    .getOrCreate())

print(f"Spark {spark.version}")

CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
YEARS = (2024, 2025)

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch(url, target):
    if target.exists():
        return "skip"
    target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, target)
    return "ok"

for y in YEARS:
    for m in range(1, 13):
        filename = f"yellow_tripdata_{y}-{m:02d}.parquet"
        status = fetch(f"{CDN_BASE}/trip-data/{filename}", DATA_DIR / filename)
        print(f"{status:>6}  {filename}")

status = fetch(f"{CDN_BASE}/misc/taxi_zone_lookup.csv", DATA_DIR / "taxi_zone_lookup.csv")
print(f"{status:>6}  taxi_zone_lookup.csv")

df_native = (spark.read.option("mergeSchema", "true")
    .parquet(*sorted(str(p) for p in DATA_DIR.glob("yellow_tripdata_*.parquet"))))

zones = (spark.read.option("header", True).option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv")))

df_raw = df_native.selectExpr(
    "cast(VendorID as tinyint) VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "cast(passenger_count as tinyint) passenger_count",
    "cast(trip_distance as float) trip_distance",
    "cast(RatecodeID as tinyint) RatecodeID",
    "store_and_fwd_flag",
    "cast(PULocationID as smallint) PULocationID",
    "cast(DOLocationID as smallint) DOLocationID",
    "cast(payment_type as tinyint) payment_type",
    "cast(fare_amount as float) fare_amount",
    "cast(extra as float) extra",
    "cast(mta_tax as float) mta_tax",
    "cast(tip_amount as float) tip_amount",
    "cast(tolls_amount as float) tolls_amount",
    "cast(improvement_surcharge as float) improvement_surcharge",
    "cast(total_amount as float) total_amount",
    "cast(congestion_surcharge as float) congestion_surcharge",
    "cast(Airport_fee as float) Airport_fee",
    "cast(cbd_congestion_fee as float) cbd_congestion_fee",
)

df_filtered = (df_raw
    .filter(F.col("tpep_pickup_datetime") >= F.lit("2024-01-01"))
    .filter(F.col("tpep_pickup_datetime") < F.lit("2026-01-01"))
    .filter(F.col("trip_distance").between(0, 200))
    .filter(F.col("fare_amount").between(0, 1000))
    .filter(F.col("total_amount").between(0, 1200))
    .filter(~((F.col("trip_distance") == 0) & (F.col("fare_amount") > 0))))

n_raw, n_filtered = df_raw.count(), df_filtered.count()
pct_removed = (n_raw - n_filtered) / n_raw * 100
print(f"Crudo: {n_raw:,} | Tras filtros: {n_filtered:,} | Pérdida: {pct_removed:.2f}%")
assert pct_removed < 15.0, f"Filtros removieron {pct_removed:.2f}% > 15%. Revisar datos o filtros."

df_clean = (df_filtered
    .withColumn("passenger_count",
        F.when(F.col("passenger_count").between(1, 6), F.col("passenger_count"))
         .otherwise(F.lit(1).cast("byte")))
    .withColumn("cbd_congestion_fee",
        F.when(F.col("cbd_congestion_fee").isNull() | (F.col("tpep_pickup_datetime") < F.lit("2025-01-05")),
               F.lit(0.0).cast("float"))
         .otherwise(F.col("cbd_congestion_fee")))
    .withColumn("congestion_surcharge", F.coalesce(F.col("congestion_surcharge"), F.lit(0.0).cast("float")))
    .withColumn("Airport_fee", F.coalesce(F.col("Airport_fee"), F.lit(0.0).cast("float")))
    .withColumn("RatecodeID", F.coalesce(F.col("RatecodeID"), F.lit(99).cast("byte")))
    .withColumn("store_and_fwd_flag", F.coalesce(F.col("store_and_fwd_flag"), F.lit("F"))))

imputed_cols = ["passenger_count", "cbd_congestion_fee", "congestion_surcharge",
                "Airport_fee", "RatecodeID", "store_and_fwd_flag"]
nulls = df_clean.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in imputed_cols]).first()
assert all((nulls[c] or 0) == 0 for c in imputed_cols), f"Nulos remanentes: {nulls.asDict()}"
print("Imputaciones aplicadas; 0 nulos en las 6 columnas objetivo.")


In [ ]:
# Construcción del estrato y recálculo del diccionario de fracciones de M.
airport_ids = {r.LocationID for r in zones.filter(F.col("service_zone").isin("Airports", "EWR")).collect()}
unknown_ids = {264, 265}
manhattan_ids = {r.LocationID for r in zones.filter(F.col("Borough") == "Manhattan").collect()} - airport_ids - unknown_ids
outer_ids = {r.LocationID for r in zones.filter(F.col("Borough").isin("Brooklyn", "Queens", "Bronx", "Staten Island")).collect()} - airport_ids - unknown_ids

df_feat = (df_clean
    .withColumn("pu_macro_zone",
        F.when(F.col("PULocationID").isin(sorted(airport_ids)), "airport")
         .when(F.col("PULocationID").isin(sorted(unknown_ids)), "unknown")
         .when(F.col("PULocationID").isin(sorted(manhattan_ids)), "manhattan")
         .when(F.col("PULocationID").isin(sorted(outer_ids)), "outer_borough")
         .otherwise("unknown"))
    .withColumn("payment_group",
        F.when(F.col("payment_type") == 0, "flex")
         .when(F.col("payment_type") == 1, "credit")
         .when(F.col("payment_type") == 2, "cash")
         .otherwise("other"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("dow", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("day_hour_bucket",
        F.when(F.col("pickup_hour").between(0, 5), "late_night")
         .when(F.col("dow").isin(1, 7), "weekend")
         .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(6, 10), "weekday_am")
         .when(F.col("dow").between(2, 6) & F.col("pickup_hour").between(16, 20), "weekday_pm_peak")
         .otherwise("other"))
    .withColumn("trip_distance_bin",
        F.when(F.col("trip_distance") < 1.12, "short")
         .when(F.col("trip_distance") < 12.43, "medium")
         .otherwise("long"))
    .withColumn("is_flex_fare", F.col("payment_type") == 0)
    .withColumn("cbd_period_flag",
        F.when(F.col("tpep_pickup_datetime") < F.lit("2025-01-05"), "pre_cbd").otherwise("post_cbd"))
    .withColumn("stratum_id",
        F.concat_ws("|",
            F.col("pu_macro_zone"), F.col("payment_group"),
            F.col("day_hour_bucket"), F.col("trip_distance_bin"))))

ESTIMATED_M = 5_030_141
N_M_TARGET = 5_000_000
MIN_FLOOR_M = 500

strata_D = (df_feat.groupBy("stratum_id").count()
    .withColumnRenamed("count", "n_D")
    .withColumn("target_n",
        F.least(F.col("n_D"),
                F.greatest(F.lit(MIN_FLOOR_M).cast("long"),
                           F.round(F.lit(N_M_TARGET) * F.col("n_D") / F.lit(n_filtered)).cast("long"))))
    .withColumn("fraction", F.col("target_n") / F.col("n_D")))

fractions = {r["stratum_id"]: float(r["fraction"]) for r in strata_D.select("stratum_id", "fraction").collect()}
assert len(fractions) == 240, f"Se esperaban 240 estratos, se obtuvieron {len(fractions)}"
assert all(0 < f <= 1.0 for f in fractions.values())

print(f"Estratos en D: {len(fractions)}")
print("Variables de estrato construidas:", sorted(set(df_feat.columns) - set(df_clean.columns)))


In [ ]:
# Extracción de M mediante sampleBy, con el diccionario de fracciones de Etapa 2.
M = df_feat.stat.sampleBy("stratum_id", fractions, seed=42).cache()
n_M = M.count()

print(f"|M| = {n_M:,} (objetivo {N_M_TARGET:,}, esperado ~5.03M)")
assert abs(n_M - ESTIMATED_M) / ESTIMATED_M < 0.02, f"|M| diverge: {n_M:,}"

M.select("stratum_id", "fare_amount", "trip_distance", "pu_macro_zone", "payment_group").show(5, truncate=False)


### 2.1 Construcción de M' a partir de M

> **Nota al profesor:** esta sección reutiliza de forma idéntica la técnica de la Actividad 3: ventana exacta estratificada con piso determinístico de 50 filas por estrato y fracción global 0.20. Se mantiene para preservar continuidad metodológica con la muestra individual que fue validada en la entrega anterior.

A partir de M se construye M' como subconjunto individual manejable. Para cada estrato `s`, se define `target_n_s = min(n_M_s, max(50, floor(0.20 * n_M_s)))`. Luego se ordenan aleatoriamente las filas dentro de cada `stratum_id` con semilla fija y se conservan las primeras `target_n_s`. Esto conserva todos los estratos y reduce el riesgo de perder perfiles raros.


In [ ]:
# Construcción de M' por ventana exacta con piso determinístico de 50.
F_GLOBAL_MP = 0.20
MIN_FLOOR_MP = 50

counts_M = M.groupBy("stratum_id").count().withColumnRenamed("count", "n_M_s")
target_Mp = counts_M.withColumn(
    "target_n_s",
    F.least(
        F.col("n_M_s"),
        F.greatest(F.lit(MIN_FLOOR_MP).cast("long"),
                   F.floor(F.lit(F_GLOBAL_MP) * F.col("n_M_s")).cast("long"))))

w_Mp = Window.partitionBy("stratum_id").orderBy(F.rand(seed=42))
M_prime = (M.join(target_Mp, "stratum_id")
    .withColumn("rn", F.row_number().over(w_Mp))
    .filter(F.col("rn") <= F.col("target_n_s"))
    .drop("rn", "target_n_s", "n_M_s")
    .cache())

n_Mp = M_prime.count()
print(f"|M'| = {n_Mp:,} (esperado ~1.0M)")
assert 950_000 <= n_Mp <= 1_100_000, f"|M'| fuera de rango: {n_Mp:,}"


### 2.2 Validación de representatividad M' vs M

> **Nota al profesor:** esta validación replica la verificación compacta de la Actividad 3. Se revisan tamaño, cobertura de estratos, piso mínimo y marginales de las cuatro variables de caracterización que definen `stratum_id`.


In [ ]:
# Validación compacta M' vs M.
n_M_v, n_Mp_v = M.count(), M_prime.count()
strata_M = M.select("stratum_id").distinct().count()
strata_Mp = M_prime.select("stratum_id").distinct().count()
min_count_Mp = M_prime.groupBy("stratum_id").count().agg(F.min("count")).first()[0]

print(f"|M| = {n_M_v:,}  |M'| = {n_Mp_v:,}  ratio = {n_Mp_v / n_M_v:.4f}")
print(f"Estratos en M = {strata_M}, en M' = {strata_Mp} (debe ser 240)")
print(f"Piso mínimo por estrato en M' = {min_count_Mp} (debe ser >= 50)\n")
assert strata_Mp == strata_M == 240, "M' perdió estratos"
assert min_count_Mp >= 50, f"Piso violado: {min_count_Mp}"

for col_name in ["pu_macro_zone", "payment_group", "day_hour_bucket", "trip_distance_bin"]:
    p_M = {r[col_name]: r["count"] / n_M_v for r in M.groupBy(col_name).count().collect()}
    p_Mp = {r[col_name]: r["count"] / n_Mp_v for r in M_prime.groupBy(col_name).count().collect()}
    max_diff_pp = max(abs(p_M.get(k, 0) - p_Mp.get(k, 0)) for k in set(p_M) | set(p_Mp)) * 100
    print(f"  {col_name}: max |p_M - p_M'| = {max_diff_pp:.4f} pp")
    assert max_diff_pp < 0.5, f"Marginal de {col_name} diverge: {max_diff_pp:.4f} pp"


## 3. Preparación del conjunto de entrenamiento y prueba

En aprendizaje no supervisado no existe una etiqueta objetivo que proteger durante la partición. Aun así, el split train/test es útil para revisar estabilidad: el modelo aprende la estructura en train y después se evalúa si el patrón se mantiene en test mediante silhouette, tamaños de cluster y perfiles agregados.

Se mantiene el split estratificado exacto de la Actividad 3 para minimizar sesgos en las variables de caracterización. La partición usa una ventana por `stratum_id`, orden aleatorio reproducible y corte `floor(0.8 * n_s)` para train; el resto queda en test. A diferencia de `randomSplit` o `sampleBy`, este método garantiza conteos determinísticos por estrato y conserva todos los perfiles raros protegidos por el piso de M'.


In [ ]:
# Split estratificado exacto 80/20 sobre M_prime.
TRAIN_RATIO = 0.8

counts_Mp = M_prime.groupBy("stratum_id").count().withColumnRenamed("count", "n_Mp_s")
w_split = Window.partitionBy("stratum_id").orderBy(F.rand(seed=123))

Mp_with_rn = (M_prime.join(counts_Mp, "stratum_id")
    .withColumn("rn", F.row_number().over(w_split))
    .withColumn("train_cutoff", F.floor(F.lit(TRAIN_RATIO) * F.col("n_Mp_s")).cast("long")))

train_df = (Mp_with_rn.filter(F.col("rn") <= F.col("train_cutoff"))
            .drop("rn", "train_cutoff", "n_Mp_s").cache())
test_df  = (Mp_with_rn.filter(F.col("rn") >  F.col("train_cutoff"))
            .drop("rn", "train_cutoff", "n_Mp_s").cache())

n_train, n_test = train_df.count(), test_df.count()
print(f"|train| = {n_train:,}  |test| = {n_test:,}  ratio_train = {n_train / (n_train + n_test):.4f}")


In [ ]:
# Verificación post-split: cobertura de estratos, piso y marginales train vs test.
strata_train = train_df.select("stratum_id").distinct().count()
strata_test = test_df.select("stratum_id").distinct().count()
min_train = train_df.groupBy("stratum_id").count().agg(F.min("count")).first()[0]
min_test = test_df.groupBy("stratum_id").count().agg(F.min("count")).first()[0]

print(f"Estratos train = {strata_train}, test = {strata_test} (esperado 240)")
print(f"Piso min en train = {min_train}, en test = {min_test} (test debe ser >= 10)\n")
assert strata_train == 240 and strata_test == 240
assert min_test >= 10, f"Piso en test violado: {min_test}"

for col_name in ["pu_macro_zone", "payment_group", "day_hour_bucket", "trip_distance_bin"]:
    p_tr = {r[col_name]: r["count"] / n_train for r in train_df.groupBy(col_name).count().collect()}
    p_te = {r[col_name]: r["count"] / n_test for r in test_df.groupBy(col_name).count().collect()}
    max_diff_pp = max(abs(p_tr.get(k, 0) - p_te.get(k, 0)) for k in set(p_tr) | set(p_te)) * 100
    print(f"  {col_name}: max |p_train - p_test| = {max_diff_pp:.4f} pp")
    assert max_diff_pp < 0.5, f"Marginal de {col_name} diverge: {max_diff_pp:.4f} pp"
